In [ ]:
%%writefile fmm.h
#pragma once           // il file viene incluso una sola volta
#include <cstdio>    
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
#include <chrono>      // misura dei tempi
#include <fstream>     // scrittura su file
const float INF = 1e30f;
typedef std::pair<float, int> Entry; // coppia (u, indice) per la coda di priorita'

__host__ __device__ float solve_quadratic(float a, float b, float F, float h)
{
    float Fh = F * h;
    if (fabsf(a - b) <= Fh)                                   
        return 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));
    return fminf(a, b) + Fh;                                  
}

float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int i, int j, int N, float h)
{
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[i * N + (j - 1)] : INF;
    float right = (j < N - 1) ? u[i * N + (j + 1)] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[(i - 1) * N + j] : INF;
    float up    = (i < N - 1) ? u[(i + 1) * N + j] : INF;
    // Per arrivare devi scorrere come un indice globale: u[i][j] = u[i * N + j]!

    float a = fminf(left, right);
    float b = fminf(down, up);

    int idx = i * N + j;
    float q = solve_quadratic(a, b, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ij, u)
}
void fmm(std::vector<float>& u, const std::vector<float>& F, int N, float h, int src)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };      // stato di ogni nodo
    std::vector<char> state(N * N, LONTANO);    // array di char NxN, inizializzato a LONTANO

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    heap.push(Entry(u[src], src));                   // la sorgente e' il primo Trial
    state[src] = ATTIVO;

    const int di[4] = {  0, 0, -1, 1 };              // spostamenti verso i 4 vicini:
    const int dj[4] = { -1, 1,  0, 0 };              // sinistra, destra, sotto, sopra

    while (!heap.empty()) {                          // while A != vuoto
        Entry trial = heap.top();                    // Trial <- radice del min-heap
        heap.pop();                                  // Rimuovi Trial da A
        int x = trial.second;                        // indice del nodo Trial
        if (state[x] == NOTO || trial.first > u[x])  // coppia "scaduta": si ignora
            continue;
        state[x] = NOTO;                             // Aggiungi Trial a Set

        int i = x / N, j = x % N;                    // riga e colonna del Trial
        for (int k = 0; k < 4; ++k) {                // foreach x_nb adiacente a Trial
            int ni = i + di[k], nj = j + dj[k];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N) continue;   // fuori dal dominio
            int nb = ni * N + nj;   // indice del nodo vicino
            if (state[nb] == NOTO) continue;         // if x_nb non appartiene a Set

            float q = solve_node(u, F, ni, nj, N, h);
            if (q < u[nb]) {                         // u(x_nb) <- soluzione di g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                  // Aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}
double relative_error(const std::vector<float>& u, int N, float h)
{
    double num = 0.0, den = 0.0;
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j) {
            double x = -1.0 + j * h, y = -1.0 + i * h;   // coordinate del nodo (i, j)
            double U = std::sqrt(x * x + y * y);        // soluzione esatta
            double d = u[i * N + j] - U;
            num += d * d;                               // ||u - U||^2
            den += U * U;                               // ||U||^2
        }
    return std::sqrt(num / den);    // Norma euclidea relativa: ||u - U|| / ||U||
}

In [ ]:
%%writefile fim.cu
#include <cstdio>          
#include <cstdlib>         
#include <cuda_runtime.h>
#include "fmm.h"           // Fast Marching

// Controllo degli errori CUDA: se la chiamata fallisce, stampa file, riga e motivo ed esce.
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)
#define TILE 8 // dimensione del blocco 8x8 = 64 thread

__device__ float solve_in_tile(const float s[TILE + 2][TILE + 2], int y, int x, float F, float h)
{
    float a = fminf(s[y][x - 1], s[y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[y - 1][x], s[y + 1][x]);               // b <- min dei vicini lungo y
    return fminf(s[y][x], solve_quadratic(a, b, F, h));      // return min(u_ij, u)
}

__global__ void update_active_tiles(const int* list, float* u, const float* F,
                                    int N, int nbx, float h, int src, int niter, float eps, unsigned char* Cb_out)
{
    __shared__ float s_u[TILE + 2][TILE + 2];                // tile + halo, nella shared memory

    int tile = list[blockIdx.x];                             // il blocco di nodi da elaborare
    int bi = tile / nbx, bj = tile % nbx;                    // riga e colonna del blocco
    int tx = threadIdx.x, ty = threadIdx.y;
    int tid = ty * TILE + tx;                                // numero del thread nel blocco: 0..63
    int i = bi * TILE + ty, j = bj * TILE + tx;              // nodo di questo thread
    bool inside = (i < N && j < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || (i * N + j == src);             // nodi da non aggiornare
    float f = inside ? F[i * N + j] : 1.0f;                  // F_ij del nodo

    // Global -> Shared: 100 celle (tile + halo) con 64 thread
    for (int k = tid; k < (TILE + 2) * (TILE + 2); k += TILE * TILE) {
        int sy = k / (TILE + 2), sx = k % (TILE + 2);       // cella (sy, sx) della shared memory
        int gi = bi * TILE + sy - 1;                         // nodo della griglia corrispondente:
        int gj = bj * TILE + sx - 1;                         // -1 perche' la riga/colonna 0 e' l'halo
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N);
        s_u[sy][sx] = in_domain ? u[gi * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();                                         // la tile e' completa prima di usarla

    // niter iterazioni locali in shared memory
    bool conv = true;                                        // Cn del nodo
    for (int it = 0; it < niter; ++it) {
        float p = s_u[ty + 1][tx + 1];                       // valore prima dell'aggiornamento
        float q = fixed ? p                                  // sorgente / fuori: non cambia
                        : solve_in_tile(s_u, ty + 1, tx + 1, f, h);
        __syncthreads();                                     // (A) tutti hanno letto i vicini...
        s_u[ty + 1][tx + 1] = q;
        __syncthreads();                                     // (B) ...e tutti hanno scritto
        conv = fabsf(p - q) < eps;                           // Cn: il nodo non e' cambiato
    }
    // Shared -> Global 100 celle ogni thread scrive il proprio nodo
    s_c[tid] = conv;
    __syncthreads();
    for (int stride = TILE * TILE / 2; stride > 0; stride /= 2) {     // 32, 16, 8, 4, 2, 1
        if (tid < stride) s_c[tid] = s_c[tid] && s_c[tid + stride];
        __syncthreads();
    }
    if (tid == 0) Cb_out[tile] = s_c[0];                     // vero solo se TUTTI i nodi sono convergenti
}
int main()
{
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d | max thread per blocco: %d | shared memory per blocco: %zu KB\n",
           prop.name, prop.multiProcessorCount, prop.maxThreadsPerBlock, prop.sharedMemPerBlock / 1024);

    const int   N = 101;
    const float h = 2.0f / (N - 1);
    std::vector<float> u_fmm(N * N, INF);
    std::vector<float> F(N * N, 1.0f);
    const int si = N / 2, sj = N / 2;
    u_fmm[si * N + sj] = 0.0f;
    fmm(u_fmm, F, N, h, si * N + sj);   // CPU: Fast Marching sequenziale
    
    const int nbx = (N + TILE - 1) / TILE;                   // N = 101 -> 13 blocchi per lato
    const int tile_src = (si / TILE) * nbx + (sj / TILE);    // il blocco che contiene la sorgente
    const float eps = 1e-6f;                                 // tolleranza di convergenza
    int* d_list; float *d_u, *d_F; unsigned char* d_Cb;
    CUDA_CHECK(cudaMalloc(&d_list, sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_u, N * N * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_F, N * N * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_Cb, nbx * nbx));                // un Cb (1 byte) per ogni blocco
    CUDA_CHECK(cudaMemcpy(d_list, &tile_src, sizeof(int), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), N * N * sizeof(float), cudaMemcpyHostToDevice));

    for (int niter : { 4, 10, 11, 16 }) {
        std::vector<float> u_fim(N * N, INF);                // condizione iniziale (eq. 4.4)
        u_fim[si * N + sj] = 0.0f;
        CUDA_CHECK(cudaMemcpy(d_u, u_fim.data(), N * N * sizeof(float), cudaMemcpyHostToDevice));
        update_active_tiles<<<1, dim3(TILE, TILE)>>>(d_list, d_u, d_F, N, nbx, h, si * N + sj,
                                                    niter, eps, d_Cb);
        CUDA_CHECK(cudaGetLastError());
        unsigned char Cb;
        CUDA_CHECK(cudaMemcpy(&Cb, d_Cb + tile_src, 1, cudaMemcpyDeviceToHost));   // solo Cb[tile_src]
        printf("niter = %2d: Cb(blocco %d) = %d\n", niter, tile_src, Cb);
    }
    CUDA_CHECK(cudaFree(d_list)); CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F)); CUDA_CHECK(cudaFree(d_Cb));

    return 0;
}


In [ ]:
!nvcc -O3 -arch=sm_75 fim.cu -o fim && ./fim